# Xarray basics

Load, inspect, subset, aggregate, resample, group, and plot an xarray dataset.

Run the cells in order. Credentials are read from local configuration; this notebook contains no saved credentials.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

try:
    import cartopy.crs as ccrs
except ImportError:
    ccrs = None


## 1. Configure the map


In [ ]:
# Set to False for a regular longitude/latitude plot.
USE_CARTOPY = True
MAP_PROJECTION = "orthographic"  # "planar" or "orthographic"

if USE_CARTOPY and ccrs is None:
    raise ImportError("Install cartopy or set USE_CARTOPY = False.")

if MAP_PROJECTION not in {"planar", "orthographic"}:
    raise ValueError("MAP_PROJECTION must be 'planar' or 'orthographic'.")


## 2. Load a NetCDF dataset


In [ ]:
ds = xr.tutorial.load_dataset("air_temperature")
ds


## 3. Inspect data and metadata


In [ ]:
print("GLOBAL METADATA")
for name, value in ds.attrs.items():
    print(f"{name}: {value}")

print("\nAIR VARIABLE METADATA")
for name, value in ds["air"].attrs.items():
    print(f"{name}: {value}")


## 4. Convert units and create subsets


In [ ]:
# Convert from Kelvin to Celsius.
temperature = ds["air"] - 273.15
temperature.attrs = {"long_name": "Air temperature", "units": "°C"}

first_map = temperature.isel(time=0)
january = temperature.sel(time=slice("2013-01-01", "2013-01-31"))

# Latitude is ordered north-to-south; 230–250° is 130–110°W.
region = temperature.sel(lat=slice(50, 30), lon=slice(230, 250))

# Select the nearest grid point to 40°N, 120°W.
my_lon, my_lat = 240, 40
point = temperature.sel(lat=my_lat, lon=my_lon, method="nearest")

print("SUBSET SIZES")
print("First map:", dict(first_map.sizes))
print("January:", dict(january.sizes))
print("Region:", dict(region.sizes))
print("Point:", dict(point.sizes))


## 5. Aggregate the data


In [ ]:
# Average across time to create a map.
mean_map = temperature.mean(dim="time", keep_attrs=True)

# Weight by latitude to account for different grid-cell areas.
weights = np.cos(np.deg2rad(region.lat))
regional_mean = region.weighted(weights).mean(
    dim=("lat", "lon"),
    keep_attrs=True,
)

print(f"Point minimum: {point.min().item():.1f} °C")
print(f"Point maximum: {point.max().item():.1f} °C")
print(f"Point mean:    {point.mean().item():.1f} °C")


## 6. Resample and group the point time series


In [ ]:
daily = point.resample(time="1D").mean(keep_attrs=True)
monthly = point.resample(time="MS").mean(keep_attrs=True)
monthly_cycle = point.groupby("time.month").mean(dim="time", keep_attrs=True)

print("FIRST SIX MONTHLY AVERAGES")
print(monthly.isel(time=slice(0, 6)).to_pandas())
print("\nMEAN ANNUAL CYCLE")
print(monthly_cycle.to_pandas())


## 7. Create the figure and map


In [ ]:
fig = plt.figure(figsize=(13, 9))

if USE_CARTOPY:
    data_crs = ccrs.PlateCarree()
    if MAP_PROJECTION == "planar":
        projection = ccrs.PlateCarree(central_longitude=-110)
    else:
        projection = ccrs.Orthographic(
            central_longitude=-110,
            central_latitude=40,
        )
    ax1 = fig.add_subplot(2, 2, 1, projection=projection)
    map_options = {"transform": data_crs}
else:
    ax1 = fig.add_subplot(2, 2, 1)
    map_options = {}

mean_map.plot(
    ax=ax1,
    cmap="seismic",
    cbar_kwargs={"label": "Air temperature (°C)", "shrink": 0.75},
    **map_options,
)

if USE_CARTOPY:
    ax1.coastlines(resolution="50m", linewidth=0.7)
    if MAP_PROJECTION == "orthographic":
        ax1.set_global()
        ax1.gridlines(linewidth=0.5, linestyle="--", color="gray", alpha=0.6)
    else:
        ax1.set_extent([-160, -30, 15, 75], crs=data_crs)
        grid = ax1.gridlines(
            draw_labels=True,
            linewidth=0.5,
            linestyle="--",
            alpha=0.6,
        )
        grid.top_labels = False
        grid.right_labels = False
else:
    ax1.grid(linestyle="--", alpha=0.5)

point_options = {"transform": data_crs} if USE_CARTOPY else {}
ax1.scatter(
    my_lon,
    my_lat,
    color="black",
    marker="o",
    s=60,
    edgecolor="white",
    linewidth=1,
    zorder=10,
    **point_options,
)
ax1.set_title("Mean air temperature")


## 8. Add time-series plots


In [ ]:
ax2 = fig.add_subplot(2, 2, 2)
daily.plot(ax=ax2, label="Daily", alpha=0.5)
monthly.plot(ax=ax2, label="Monthly", linewidth=2)
ax2.set_title("Point temperature: nearest to 40°N, 120°W")
ax2.legend()
ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(2, 2, 3)
monthly_cycle.plot(ax=ax3, marker="o")
ax3.set_title("Mean annual cycle at the point")
ax3.set_xticks(range(1, 13))
ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(2, 2, 4)
regional_mean.plot(ax=ax4)
ax4.set_title("Area-weighted regional temperature")
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Close the dataset


In [ ]:
ds.close()
